# EMIS 災情類別分類

使用 2006–2013 年 EMIS 資料，比較多層感知器（MLP）與隨機森林（Random Forest）的災情類別分類結果。

In [ ]:
%pip install -q pandas openpyxl scikit-learn

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
SAMPLE_FRACTION = 0.5
MIN_CLASS_SIZE = 100
FEATURE_COLUMNS = ["縣市", "災情細項", "災情描述"]
TARGET_COLUMN = "災情類別"
DATA_URL = (
    "https://raw.githubusercontent.com/xixa3333/"
    "Applying-Machine-Learning-to-Public-Internet-of-Things/"
    "main/data/EMIS_2006-2013.xlsx"
)

In [ ]:
def find_local_data():
    candidates = [
        Path("data/EMIS_2006-2013.xlsx"),
        Path("../data/EMIS_2006-2013.xlsx"),
    ]
    return next((path for path in candidates if path.exists()), None)


def load_and_prepare_data():
    data_source = find_local_data() or DATA_URL
    data = pd.read_excel(data_source, sheet_name="工作表1")
    data = data.dropna(subset=[TARGET_COLUMN]).copy()
    data[FEATURE_COLUMNS] = data[FEATURE_COLUMNS].fillna("缺失值").astype(str)

    sampled, _ = train_test_split(
        data,
        train_size=SAMPLE_FRACTION,
        stratify=data[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )

    class_counts = sampled[TARGET_COLUMN].value_counts()
    classes_to_keep = class_counts[class_counts >= MIN_CLASS_SIZE].index
    sampled = sampled[sampled[TARGET_COLUMN].isin(classes_to_keep)].copy()

    label_encoder = LabelEncoder()
    X = sampled[FEATURE_COLUMNS]
    y = label_encoder.fit_transform(sampled[TARGET_COLUMN])
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    return X_train, X_test, y_train, y_test, label_encoder


def make_preprocessor():
    return ColumnTransformer(
        [("categorical", OneHotEncoder(handle_unknown="ignore"), FEATURE_COLUMNS)]
    )


def evaluate(model, X_test, y_test, label_encoder):
    predictions = model.predict(X_test)
    labels = sorted(set(y_test) | set(predictions))
    target_names = label_encoder.inverse_transform(labels)
    print(f"準確率: {accuracy_score(y_test, predictions):.4f}")
    print(
        classification_report(
            y_test, predictions, labels=labels, target_names=target_names, zero_division=0
        )
    )


X_train, X_test, y_train, y_test, label_encoder = load_and_prepare_data()
print(f"訓練資料: {len(X_train):,} 筆；測試資料: {len(X_test):,} 筆")

## 多層感知器（MLP）

In [ ]:
mlp = Pipeline(
    [
        ("preprocess", make_preprocessor()),
        ("scale", StandardScaler(with_mean=False)),
        (
            "model",
            MLPClassifier(
                hidden_layer_sizes=(100,),
                max_iter=100,
                early_stopping=True,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
mlp.fit(X_train, y_train)
evaluate(mlp, X_test, y_test, label_encoder)

## 隨機森林（Random Forest）

In [ ]:
random_forest = Pipeline(
    [
        ("preprocess", make_preprocessor()),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
random_forest.fit(X_train, y_train)
evaluate(random_forest, X_test, y_test, label_encoder)